# Lesson 01: Introduction to Computer Vision — Python (Colab)

This notebook implements the core ideas from Lesson 01: representing images as sampled/quantized signals, working with channels and luma, applying point-wise intensity transforms and histograms, and performing convolution/correlation with common kernels.

**How to use:** Run cells top-to-bottom. Try the suggested experiments at the end to build intuition.

## Setup

This notebook uses NumPy and Matplotlib (preinstalled in Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1) Image as a sampled, quantized signal (synthetic grayscale)
H, W = 64, 64
y, x = np.mgrid[0:H, 0:W]
img = (0.6*np.sin(2*np.pi*x/16) + 0.4*np.cos(2*np.pi*y/10))
img = ((img - img.min())/(img.max()-img.min()) * 255).astype(np.uint8)
print('shape:', img.shape, 'dtype:', img.dtype, 'min/max:', img.min(), img.max())

# 2) RGB channels and luma (perceived grayscale)
rgb = np.zeros((H, W, 3), dtype=np.uint8)
rgb[..., 0] = np.clip(img + 20, 0, 255)       # R
rgb[..., 1] = np.clip(img, 0, 255)            # G
rgb[..., 2] = np.clip(img//2, 0, 255)         # B
luma = (0.299*rgb[...,0] + 0.587*rgb[...,1] + 0.114*rgb[...,2]).astype(np.uint8)
naive = rgb.mean(axis=2).astype(np.uint8)

# 3) Intensity transforms: g = clip(alpha*I + beta), gamma
def linear_transform(I, alpha=1.2, beta=-10):
    return np.clip(alpha*I.astype(np.float32) + beta, 0, 255).astype(np.uint8)

def gamma_transform(I, gamma=1.5):
    In = I.astype(np.float32)/255.0
    return np.clip((In**gamma)*255, 0, 255).astype(np.uint8)

bright = linear_transform(luma, alpha=1.0, beta=40)
contrast = linear_transform(luma, alpha=1.4, beta=-30)
gamma_dark = gamma_transform(luma, gamma=1.8)

# 4) Histogram
def show_hist(ax, I, title):
    ax.hist(I.ravel(), bins=256, range=(0,255), color='#1f5f7a', alpha=0.8)
    ax.set_title(title); ax.set_xlim(0,255)

# 5) Convolution (correlation-style sliding) and a flipped kernel for true convolution
def convolve2d(I, K, flip=True):
    Kh, Kw = K.shape
    pad_h, pad_w = Kh//2, Kw//2
    Kf = np.flipud(np.fliplr(K)) if flip else K
    P = np.pad(I.astype(np.float32), ((pad_h,pad_h),(pad_w,pad_w)), mode='edge')
    out = np.zeros_like(I, dtype=np.float32)
    for i in range(I.shape[0]):
        for j in range(I.shape[1]):
            patch = P[i:i+Kh, j:j+Kw]
            out[i,j] = np.sum(patch * Kf)
    return out

box = np.ones((3,3), np.float32)/9.0
gauss = (1/16.0)*np.array([[1,2,1],[2,4,2],[1,2,1]], np.float32)
sharpen = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], np.float32)
sobelx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], np.float32)

blurred = np.clip(convolve2d(luma, box), 0, 255).astype(np.uint8)
sharp = np.clip(convolve2d(luma, sharpen), 0, 255).astype(np.uint8)
edges = convolve2d(luma, sobelx)
edges_vis = np.clip(np.abs(edges), 0, 255).astype(np.uint8)

# correlation vs convolution for an asymmetric 1D kernel (concept)
signal = np.array([1,2,3,4,3,2,1], np.float32)
k = np.array([1,0,-1], np.float32)
corr = np.array([np.sum(signal[i:i+3]*k) for i in range(len(signal)-2)])
conv = np.array([np.sum(signal[i:i+3]*k[::-1]) for i in range(len(signal)-2)])
print('1D correlation:', corr, '  1D convolution:', conv)

# 6) Visualize
fig, axs = plt.subplots(3, 4, figsize=(14,9))
axs[0,0].imshow(img, cmap='gray'); axs[0,0].set_title('Grayscale signal')
axs[0,1].imshow(rgb); axs[0,1].set_title('RGB')
axs[0,2].imshow(luma, cmap='gray'); axs[0,2].set_title('Luma (weighted)')
axs[0,3].imshow(naive, cmap='gray'); axs[0,3].set_title('Naive average')
axs[1,0].imshow(bright, cmap='gray'); axs[1,0].set_title('Bright (β=+40)')
axs[1,1].imshow(contrast, cmap='gray'); axs[1,1].set_title('Contrast (α=1.4)')
axs[1,2].imshow(gamma_dark, cmap='gray'); axs[1,2].set_title('Gamma (γ=1.8)')
show_hist(axs[1,3], luma, 'Histogram (luma)')
axs[2,0].imshow(blurred, cmap='gray'); axs[2,0].set_title('Box blur')
axs[2,1].imshow(sharp, cmap='gray'); axs[2,1].set_title('Sharpen')
axs[2,2].imshow(edges_vis, cmap='gray'); axs[2,2].set_title('Sobel X (|resp|)')
show_hist(axs[2,3], contrast, 'Histogram (contrast)')
for ax in axs.ravel():
    if ax.images: ax.axis('off')
plt.tight_layout(); plt.show()

## Experiments

change kernel values, bit depth/quantization (<code>(img//(256//levels))*(256//levels)</code>), or replace the synthetic image with an uploaded photo (<code>from PIL import Image; np.array(Image.open('photo.jpg').convert('L'))</code>).